# Testing chunking

In [1]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate
import mplhep as hep
# from helpers import merge_bkg, merge_cutflows

In [2]:
nfiles = 10
fmulxy1 = ["4Mu_500GeV_1p2GeV_0p019mm"]
channels = ["bkg_base_2mulj"]

In [3]:
fileset41 = utilities.make_fileset(fmulxy1, "llpNanoAOD_v2", max_files = nfiles, location_cfg="signal_4mu_v10.yaml")

In [4]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

Connection method: Direct,
Dashboard: /user/joaquin.siado.castaneda@cern.ch/proxy/8787/status,
Comm: tls://192.168.202.21:8786,Workers: 0
Dashboard: /user/joaquin.siado.castaneda@cern.ch/proxy/8787/status,Total threads: 0
Started: 3 hours ago,Total memory: 0 B


In [5]:
runner = processor.Runner(
    # executor=processor.FuturesExecutor(),              # for testing locally
    # executor=processor.IterativeExecutor(),              # for testing locally
    executor=processor.DaskExecutor(client=client),      # for dask
    # schema=NanoAODSchema,
    schema = llpnanoaodschema.LLPNanoAODSchema,
    maxchunks=1,
    skipbadfiles=True,
    # verbose=True
)

p = sidm_processor.SidmProcessor(    channels,    ["BKG_study"],    unweighted_hist=True, verbose=False)

In [ ]:
out41 = runner.run(fileset41, treename="Events", processor_instance=p)
out41 = out41["out"]
# coffea.util.save(out41, "outputs/bkg_"+ vr + "_fmu1.coffea")

Output()

In [ ]:
vamos mijo

In [ ]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate
import mplhep as hep
# from helpers import merge_bkg, merge_cutflows

In [ ]:
# nfiles = 100
sig = ["4Mu_500GeV_0p25GeV_0p004mm"]
dyj = ["DYJetsToMuMu_M10to50", "DYJetsToMuMu_M50",]
qcd = ["QCD_Pt15To20", "QCD_Pt20To30", "QCD_Pt30To50", "QCD_Pt50To80", "QCD_Pt80To120", "QCD_Pt120To170", "QCD_Pt170To300", "QCD_Pt300To470", "QCD_Pt470To600",
       "QCD_Pt600To800", "QCD_Pt800To1000", "QCD_Pt1000"]

In [ ]:
# fileset = utilities.make_fileset(fmulxy1,  "llpNanoAOD_v2",     max_files=nfiles,    location_cfg="signal_4mu_v10.yaml")
# fileset = utilities.make_fileset(djy, "skimmed_llpNanoAOD_v2", max_files = -1, location_cfg = "backgrounds.yaml",)
fileset = utilities.make_fileset(qcd, "skimmed_llpNanoAOD_v2", max_files = -1, location_cfg = "backgrounds.yaml",)



In [ ]:
import time
import uproot

good_files = []
bad_files = []

for dataset, content in fileset.items():

    files = content["files"]

    print("\nDataset:", dataset)
    print("Number of files:", len(files))

    for i, f in enumerate(files):
        print(f"\n[{i+1}/{len(files)}] testing:", f)

        try:
            t0 = time.time()

            file = uproot.open(f, timeout=30)
            tree = file["Events"]

            # minimal read test
            tree.arrays(entry_start=0, entry_stop=10)

            print(f"   OK | time={time.time()-t0:.2f}s")
            good_files.append(f)

        except Exception as e:
            print("   FAIL:", repr(e))
            bad_files.append(f)

print("\n====================")
print("GOOD FILES:", len(good_files))
print("BAD FILES:", len(bad_files))

print("\nBAD FILE LIST:")
for f in bad_files:
    print(f)

In [ ]:
jopo

In [ ]:
import time
import uproot
import awkward as ak
from coffea.nanoevents import NanoEventsFactory

treename = "Events"

all_files = []
for ds, flist in fileset41.items():
    all_files.extend(flist)

print("Total files:", len(all_files))

good_files = []
bad_files = []

for i, f in enumerate(all_files):
    print(f"\n[{i+1}/{len(all_files)}] testing:", f)

    try:
        t0 = time.time()

        # STEP 1: open file with uproot directly (safe check)
        file = uproot.open(f, timeout=30)

        # STEP 2: access tree
        tree = file[treename]

        # STEP 3: read small entry range only
        arrays = tree.arrays(entry_start=0, entry_stop=10)

        dt = time.time() - t0

        print(f"   OK | time={dt:.2f}s")
        good_files.append(f)

    except Exception as e:
        print("   FAIL:", repr(e))
        bad_files.append(f)

print("\n====================")
print("GOOD FILES:", len(good_files))
print("BAD FILES:", len(bad_files))

print("\nBAD FILE LIST:")
for f in bad_files:
    print(f)

In [ ]:
from coffea.nanoevents import NanoEventsFactory
import traceback
import time

treename = "Events"

# flatten fileset into a simple list of files
all_files = []
for ds, flist in fileset41.items():
    all_files.extend(flist)

print("Total files:", len(all_files))

good_files = []
bad_files = []

for i, f in enumerate(all_files):
    print(f"\n[{i+1}/{len(all_files)}] testing:", f)

    try:
        t0 = time.time()

        events = NanoEventsFactory.from_root(
            f,
            treepath=treename,
            schemaclass=llpnanoaodschema.LLPNanoAODSchema,
        ).events()

        n = len(events)
        dt = time.time() - t0

        print(f"   OK  | entries={n} | time={dt:.2f}s")
        good_files.append(f)

    except Exception as e:
        print("   FAIL:", repr(e))
        bad_files.append(f)

print("\nSUMMARY")
print("Good files:", len(good_files))
print("Bad files:", len(bad_files))

print("\nBAD FILE LIST:")
for f in bad_files:
    print(f)

In [ ]:
import os
import sys
import importlib

from coffea import processor
from coffea.nanoevents import NanoAODSchema
import coffea.util

import awkward as ak
import matplotlib.pyplot as plt

from dask.distributed import Client

# your local package
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path:
    sys.path.insert(1, sidm_path)

from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema

importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)

In [ ]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

In [ ]:
runner = processor.Runner(
    executor=processor.DaskExecutor(
        client=client,
        uproot_options={"timeout": 60}
    ),
    schema=llpnanoaodschema.LLPNanoAODSchema,
    skipbadfiles=True,
    maxchunks=1,
)

In [ ]:
channels = ["bkg_base",]
p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"],
    unweighted_hist=True,
    verbose=True
)

In [ ]:
import types

_old_process = sidm_processor.SidmProcessor.process

def debug_process(self, events):
    print("➡️ start process, nEvents =", len(events))

    out = _old_process(self, events)

    print("⬅️ end process")
    return out

sidm_processor.SidmProcessor.process = debug_process

In [ ]:
vr = "test"
nfiles = 10

fmulxy1 = ["4Mu_500GeV_0p25GeV_0p004mm"]

fileset41 = utilities.make_fileset(
    fmulxy1,
    "llpNanoAOD_v2",
    max_files=nfiles,
    location_cfg="signal_4mu_v10.yaml"
)

In [ ]:
out41 = runner.run(
    fileset41,
    treename="Events",
    processor_instance=p
)

out41 = out41["out"]

In [ ]:
jopo

In [ ]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate
import mplhep as hep
# from helpers import merge_bkg, merge_cutflows

In [ ]:
vr = "test"
nfiles = 5

channels = ["bkg_base",]
fmulxy1 = ["4Mu_500GeV_0p25GeV_0p004mm"]

bkgttj = ["TTJets"]
bkgdyj1 = ["DYJetsToMuMu_M10to50",]
bkgdyj2 = ["DYJetsToMuMu_M50",]
bkgqcd1 = ["QCD_Pt15To20",]
bkgqcd2 = ["QCD_Pt20To30"]
bkgqcd3 = ["QCD_Pt30To50",]
bkgqcd4 = ["QCD_Pt50To80"]
bkgqcd5 = ["QCD_Pt80To120"]
bkgqcd6 = ["QCD_Pt120To170"]
bkgqcd7 = ["QCD_Pt170To300",]
bkgqcd8 = ["QCD_Pt300To470"]
bkgqcd9 = ["QCD_Pt470To600"]
bkgqcd10 = ["QCD_Pt600To800"]
bkgqcd11 = ["QCD_Pt800To1000",] 
bkgqcd12 = ["QCD_Pt1000"]

channels = ["bkg_base",]

In [ ]:
vr = "test"
nfiles = 10

fmulxy1 = ["4Mu_500GeV_0p25GeV_0p004mm"]

client = scaleout.make_dask_client("tls://localhost:8786")
client
runner = processor.Runner(
    executor=processor.DaskExecutor(client=client),
    schema=llpnanoaodschema.LLPNanoAODSchema,
    skipbadfiles=True,
    maxchunks=1,
)

In [ ]:
import types

# save original method from your already-imported module
_old_process = sidm_processor.SidmProcessor.process

def debug_process(self, events):
    print("➡️ start process, nEvents =", len(events))

    out = _old_process(self, events)

    print("⬅️ end process")
    return out

# attach patched method to the class
sidm_processor.SidmProcessor.process = debug_process

In [ ]:
p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"],
    unweighted_hist=True, verbose=True
)

In [ ]:
fileset41 = utilities.make_fileset(
    fmulxy1,
    "llpNanoAOD_v2",
    max_files=nfiles,
    location_cfg="signal_4mu_v10.yaml"
)

In [ ]:
out41 = runner.run(
    fileset41,
    treename="Events",
    processor_instance=p
)

out41 = out41["out"]

In [ ]:
vr = "test"
nfiles = 10


fmulxy1 = ["4Mu_500GeV_0p25GeV_0p004mm"]

client = scaleout.make_dask_client("tls://localhost:8786")
client

runner = processor.Runner(
    executor=processor.DaskExecutor(client=client),
    schema = llpnanoaodschema.LLPNanoAODSchema,
    maxchunks=1,
    # maxchunks = 1 
    skipbadfiles=True,
    def process(self, events):
    print("➡️ start process, nEvents =", len(events))

    # your existing code here

    print("⬅️ end process")
    return out
)

p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"],
    unweighted_hist=True, verbose=True
)

fileset41 = utilities.make_fileset(fmulxy1, "llpNanoAOD_v2", max_files = nfiles, location_cfg="signal_4mu_v10.yaml")
out41 = runner.run(fileset41, treename="Events", processor_instance=p)
out41 = out41["out"]
coffea.util.save(out41, "outputs/bkg_"+ vr + "_fmu1.coffea")

In [ ]:
# process DYJ2
filesetdyj2 = utilities.make_fileset(bkgdyj2, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outdyj2     = runner.run(filesetdyj2, treename="Events", processor_instance=p)
outdyj2     = outdyj2["out"]
coffea.util.save(outdyj2, "outputs/bkg_" + vr + "_dyj2.coffea")

In [ ]:
# process qcd2
filesetqcd2    = utilities.make_fileset(bkgqcd2, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd2        = runner.run(filesetqcd2, treename="Events", processor_instance=p)
outqcd2        = outqcd2["out"]
coffea.util.save(outqcd2, "outputs/bkg_" + vr + "_qcd2.coffea")

In [ ]:
# process qcd3
filesetqcd3    = utilities.make_fileset(bkgqcd3, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd3        = runner.run(filesetqcd3, treename="Events", processor_instance=p)
outqcd3        = outqcd3["out"]
coffea.util.save(outqcd3, "outputs/bkg_" + vr + "_qcd3.coffea")

In [ ]:
# process qcd4
filesetqcd4    = utilities.make_fileset(bkgqcd4, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd4        = runner.run(filesetqcd4, treename="Events", processor_instance=p)
outqcd4        = outqcd4["out"]
coffea.util.save(outqcd4, "outputs/bkg_" + vr + "_qcd4.coffea")

In [ ]:
# process qcd5
filesetqcd5    = utilities.make_fileset(bkgqcd5, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd5        = runner.run(filesetqcd5, treename="Events", processor_instance=p)
outqcd5        = outqcd5["out"]
coffea.util.save(outqcd5, "outputs/bkg_" + vr + "_qcd5.coffea")

In [ ]:
# process qcd6
filesetqcd6    = utilities.make_fileset(bkgqcd6, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd6        = runner.run(filesetqcd6, treename="Events", processor_instance=p)
outqcd6        = outqcd6["out"]
coffea.util.save(outqcd6, "outputs/bkg_" + vr + "_qcd6.coffea")

In [ ]:
# process qcd7
filesetqcd7    = utilities.make_fileset(bkgqcd7, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd7        = runner.run(filesetqcd7, treename="Events", processor_instance=p)
outqcd7        = outqcd7["out"]
coffea.util.save(outqcd7, "outputs/bkg_" + vr + "_qcd7.coffea")

In [ ]:
# process qcd8
filesetqcd8    = utilities.make_fileset(bkgqcd8, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd8        = runner.run(filesetqcd8, treename="Events", processor_instance=p)
outqcd8        = outqcd8["out"]
coffea.util.save(outqcd8, "outputs/bkg_" + vr + "_qcd8.coffea")

In [ ]:
# process qcd9
filesetqcd9    = utilities.make_fileset(bkgqcd9, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd9        = runner.run(filesetqcd9, treename="Events", processor_instance=p)
outqcd9        = outqcd9["out"]
coffea.util.save(outqcd9, "outputs/bkg_" + vr + "_qcd9.coffea")

In [ ]:
# process qcd10
filesetqcd10    = utilities.make_fileset(bkgqcd10, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd10        = runner.run(filesetqcd10, treename="Events", processor_instance=p)
outqcd10        = outqcd10["out"]
coffea.util.save(outqcd10, "outputs/bkg_" + vr + "_qcd10.coffea")

In [ ]:
# process qcd11
filesetqcd11    = utilities.make_fileset(bkgqcd11, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd11        = runner.run(filesetqcd11, treename="Events", processor_instance=p)
outqcd11        = outqcd11["out"]
coffea.util.save(outqcd11, "outputs/bkg_" + vr + "_qcd11.coffea")

In [ ]:
# process qcd12
filesetqcd12    = utilities.make_fileset(bkgqcd12, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd12        = runner.run(filesetqcd12, treename="Events", processor_instance=p)
outqcd12        = outqcd12["out"]
coffea.util.save(outqcd12, "outputs/bkg_" + vr + "_qcd12.coffea")

In [ ]:
# Run only once
process = [
    # "tmu1",
    "fmu1",
    "ttj",
    "dyj1", 
    "dyj2",
    "qcd1",
    "qcd2",
    "qcd3",
    "qcd4",
    "qcd5",
    "qcd6",
    "qcd7",
    "qcd8",
    "qcd9",
    "qcd10",
    "qcd11",
    "qcd12",
]

outall = {}

for p in process:
    outp = coffea.util.load(f"outputs/bkg_{vr}_{p}.coffea")
    outall |= outp

coffea.util.save(outall, f"outputs/bkg_{vr}.coffea")
print(f"output file bkg_{vr}.coffea saved")

# out2mu = coffea.util.load(f"outputs/bkg_{vr}_2mu.coffea")
# out4mu = coffea.util.load(f"outputs/bkg_{vr}_4mu.coffea")
# outdyj1 = coffea.util.load(f"outputs/bkg_{vr}_dyj1.coffea")
# outdyj2 = coffea.util.load(f"outputs/bkg_{vr}_dyj2.coffea")
# outttj = coffea.util.load(f"outputs/bkg_{vr}_ttj.coffea")
# outqcd1 = coffea.util.load(f"outputs/bkg_{vr}_qcd1.coffea")
# outqcd2 = coffea.util.load(f"outputs/bkg_{vr}_qcd2.coffea")
# outqcd3 = coffea.util.load(f"outputs/bkg_{vr}_qcd3.coffea")
# outqcd4 = coffea.util.load(f"outputs/bkg_{vr}_qcd4.coffea")
# outqcd5 = coffea.util.load(f"outputs/bkg_{vr}_qcd5.coffea")
# outqcd6 = coffea.util.load(f"outputs/bkg_{vr}_qcd6.coffea")
# outqcd7 = coffea.util.load(f"outputs/bkg_{vr}_qcd7.coffea")
# outqcd8 = coffea.util.load(f"outputs/bkg_{vr}_qcd8.coffea")
# outqcd9 = coffea.util.load(f"outputs/bkg_{vr}_qcd9.coffea")
# outqcd10 = coffea.util.load(f"outputs/bkg_{vr}_qcd10.coffea")
# outqcd11 = coffea.util.load(f"outputs/bkg_{vr}_qcd11.coffea")
# outqcd12 = coffea.util.load(f"outputs/bkg_{vr}_qcd12.coffea")

# outall = {**out2mu, **out4mu, **outttj,
#           **outdyj1, **outdyj2,
#           **outqcd1, **outqcd2, **outqcd3, **outqcd4, **outqcd5, **outqcd6, **outqcd7, **outqcd8, **outqcd9, **outqcd10, **outqcd11, **outqcd12
#          }

# coffea.util.save(outall, f"outputs/bkg_{vr}.coffea")

In [ ]:
Stop mijo

# Plots

In [ ]:
vr = "31"
output = coffea.util.load(f"outputs/bkg_{vr}.coffea")

In [ ]:
# for k in output:
#     print(k)

In [ ]:
tmulxy1 = ["2Mu2E_500GeV_0p25GeV_0p004mm", "2Mu2E_500GeV_0p25GeV_0p04mm", "2Mu2E_500GeV_0p25GeV_0p4mm", "2Mu2E_500GeV_0p25GeV_2p0mm", "2Mu2E_500GeV_0p25GeV_4p0mm"]

#     "2Mu2E_500GeV_1p2GeV_0p019mm",  "2Mu2E_500GeV_1p2GeV_0p19mm",  "2Mu2E_500GeV_1p2GeV_1p9mm",  "2Mu2E_500GeV_1p2GeV_9p6mm",  "2Mu2E_500GeV_1p2GeV_19p0mm",
#     "2Mu2E_500GeV_5p0GeV_0p08mm",   "2Mu2E_500GeV_5p0GeV_0p08mm",  "2Mu2E_500GeV_5p0GeV_8p0mm",  "2Mu2E_500GeV_5p0GeV_40p0mm", "2Mu2E_500GeV_5p0GeV_80p0mm",
#     "2Mu2E_200GeV_1p2GeV_4p8mm",    "2Mu2E_500GeV_1p2GeV_1p9mm",   "2Mu2E_800GeV_1p2GeV_1p2mm",  "2Mu2E_1000GeV_1p2GeV_0p96mm",
# ]

fmulxy1 = ["4Mu_500GeV_0p25GeV_0p004mm",  "4Mu_500GeV_0p25GeV_0p04mm",   "4Mu_500GeV_0p25GeV_0p4mm",   "4Mu_500GeV_0p25GeV_2p0mm",   "4Mu_500GeV_0p25GeV_4p0mm"]

#     "4Mu_500GeV_1p2GeV_0p019mm",   "4Mu_500GeV_1p2GeV_0p19mm",    "4Mu_500GeV_1p2GeV_1p9mm",    "4Mu_500GeV_1p2GeV_9p6mm",    "4Mu_500GeV_1p2GeV_19p0mm", 
#     "4Mu_500GeV_5p0GeV_0p08mm",    "4Mu_500GeV_5p0GeV_0p8mm",     "4Mu_500GeV_5p0GeV_8p0mm",    "4Mu_500GeV_5p0GeV_40p0mm",   "4Mu_500GeV_5p0GeV_80p0mm",
#     "4Mu_200GeV_1p2GeV_4p8mm",     "4Mu_500GeV_1p2GeV_1p9mm",     "4Mu_800GeV_1p2GeV_1p2mm",    "4Mu_1000GeV_1p2GeV_0p96mm",
# ]

bkgttj = ["TTJets"]

bkgdyj = ["DYJetsToMuMu_M10to50", "DYJetsToMuMu_M50"]

bkgqcd = ["QCD_Pt15To20", "QCD_Pt20To30", "QCD_Pt30To50", "QCD_Pt50To80", "QCD_Pt80To120", "QCD_Pt120To170", "QCD_Pt170To300", "QCD_Pt300To470", 
          "QCD_Pt470To600", "QCD_Pt600To800", "QCD_Pt800To1000", "QCD_Pt1000"]

channels = ["baseNoLj",
            "bkg_base", 
            "bkg_base_iso",
            "bkg_base_iso_disp",
            "bkg_base_iso_disp_2mu2e",
            "bkg_base_iso_disp_2mu2e_dphi",
            "bkg_base_iso_disp_4mu",
            "bkg_base_iso_disp_4mu_dphi",
]

ch1 = channels[0]
ch2 = channels[1]
ch3 = channels[2]
ch4 = channels[3]
ch5 = channels[4]
ch6 = channels[5]
ch7 = channels[6]
ch8 = channels[7]


cha2name = ["Base", "Iso", "Disp", "2mu2e", r"$\Delta\Phi > 2$"]
cha2 = [ch2, ch3, ch4, ch5, ch6,]

cha4name = ["Base", "Iso", "Disp", "4mu", r"$\Delta\Phi > 2$"]
cha4 = [ch2, ch3, ch4, ch7, ch8,]

histoplot = ["lj_lj_invmass"]

lablxy1 = ["0p3", "3p0", "30", "150", "300"]
colsig=["black", "darkviolet", "navy", "magenta", "darkorange", "darkblue"]
labmx=[r"$2\mu2e: m_{xx}=200$", r"$2\mu2e: m_{xx}=500$", r"$2\mu2e: m_{xx}=800$", r"$2\mu2e: m_{xx}=1000$"]

figh, figw = 12, 10


## Looking at how each selection affect each signal and background process separately

In [ ]:
#plot process only
# DYJ = merge_bkg(output, bkgdyj, htp, ch)
# QCD = merge_bkg(output, bkgqcd, htp, ch)
# TTJ = merge_bkg(output, bkgttj, htp, ch)

# process = [tmulxy1[2], bkgttj, ]
# for ip, pr in enumerate(process):
#     for ik, htp in enumerate(histoplot):
#         plt.subplots(1, 1, figsize=(16, 10))
#         for ij, ch in enumerate(cha2):
#             if "Mu" in pr:
#                 utilities.plot(output[pr]["hists"][htp][ch, :], label = cha2name[ij])
#             else:
#                 bkg = merge_bkg(output, pr, htp, ch)
#                 utilities.plot(bkg, label = cha2name[ij])
#         # plt.title(name2mu[ik])
#         # plt.ylabel("Events")
#         # plt.yscale("log")
#         plt.legend(title=r"Selection:", alignment="left", loc=0)
#     #     plt.savefig(f"clean_plots/ch5_bkg_{vr}_lxy_{ht}_.pdf", bbox_inches="tight", dpi=300)
#     # print()

#tmu signal
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha2):
        utilities.plot(output[tmulxy1[2]]["hists"][htp][ch, :], label = cha2name[ij])
    plt.title("Signal")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)
    plt.text(0.05, 0.95,
        "m$_{XX}$ = 500 GeV\nm$_{Z_D}$ = 0.25 GeV\n$l_{xy}$ = 30cm",
        transform=plt.gca().transAxes,
        fontsize=24,
        va="top",)

#TTJets
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha2):
        TTJ = merge_bkg(output, bkgttj, htp, ch)
        utilities.plot(TTJ, label = cha2name[ij])
    plt.title("TTJets")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)

#DYJets
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha2):
        DYJ = merge_bkg(output, bkgdyj, htp, ch)
        utilities.plot(DYJ, label = cha2name[ij])
    plt.title("DYJets")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)

#DYJets
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha2):
        QCD = merge_bkg(output, bkgqcd, htp, ch)
        utilities.plot(QCD, label = cha2name[ij])
    plt.title("QCD")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)

## Stacking Backgrounds for different selections for $m_{xx} = 500$, $m_{z_D} = 0.25$ and changing $l_{xy}$

In [ ]:
###################################################
####### Stacking Backgrounds diff selections
for htp in histoplot:  
    for ic, ch in enumerate(cha2):
        
        fig, ax = plt.subplots(figsize=(figh, figw))
        
        DYJ = merge_bkg(output, bkgdyj, htp, ch)
        QCD = merge_bkg(output, bkgqcd, htp, ch)
        TTJ = merge_bkg(output, bkgttj, htp, ch)
        
        # Backgrounds
        hep.histplot([DYJ, QCD, TTJ],
                     stack=True,  histtype="fill", 
                     color=["#FFD700", "#3498DB", "#E74C3C"],
                     label=["DYJ", "QCD", "TTJ"],
                     ax=ax, )
        
        # Signal        
        for ix, sss in enumerate(tmulxy1):
            hep.histplot(output[sss]["hists"][htp][ch, ::2j],  color=colsig[ix], label=lablxy1[ix], histtype="step", linewidth=3, ax=ax)
        hep.cms.label(data=False, lumi=None, com=13, ax=ax)
        # ax.set_ylim(1, 1000000)
        # ax.yscale(1,10000)
        ax.set_yscale("log")
        ax.legend(title="Process", ncol=2)
        plt.title(f"selection: {cha2name[ic]}")
        # plt.savefig(f"clean_plots/fin_bkg_stack_{vr}_{ht}_{cha2name[ic]}_mxx.pdf", bbox_inches="tight", dpi=300)

## More LJ variables for background and signal for final cuts

In [ ]:
mhtoplot = ["lj_pt", "lj_pfMuon_pt"]
mhtoname = [r"Lepton-jet $p_T$", "Lepton-jet muon $p_T$"]
proname = [r"$2\mu2e"]
npl = 2

for ih, htp in enumerate(mhtoplot):
    plt.subplots(1, 2, figsize=(npl*figh, figw))
    plt.subplot(1,2,1)
    utilities.plot(output[tmulxy1[2]]["hists"][htp][ch6, ::2j], label = r"$2\mu2e$", color = "black")
    
    DYJ = merge_bkg(output, bkgdyj, htp, ch6)
    utilities.plot(DYJ, label="DYJ", color="#FFD700")
    
    TTJ = merge_bkg(output, bkgttj, htp, ch6)
    utilities.plot(TTJ, label="TTJ", color="#E74C3C")

    QCD = merge_bkg(output, bkgqcd, htp, ch6)
    utilities.plot(QCD, label="QCD", color="#3498DB")
    
    plt.yscale("log")
    plt.legend(title="Selections", alignment="left", loc=0)
    plt.title(mhtoname[ih])

    
    plt.subplot(1,2,2)
    utilities.plot(output[fmulxy1[2]]["hists"][htp][ch8, ::2j], label = r"$4\mu$", color = "black")

    DYJ = merge_bkg(output, bkgdyj, htp, ch8)
    utilities.plot(DYJ, label="DYJ", color="#FFD700")
    
    TTJ = merge_bkg(output, bkgttj, htp, ch8)
    utilities.plot(TTJ, label="TTJ", color="#E74C3C")

    QCD = merge_bkg(output, bkgqcd, htp, ch8)
    utilities.plot(QCD, label="QCD", color="#3498DB")
    
    plt.yscale("log")
    plt.legend(title="Selections", alignment="left", loc=0)
    plt.title(mhtoname[ih])

## Final yield summary

In [ ]:
chan = ch6        # Change this once
dphi = "LJ-LJ dPhi > 2"

print(f"{'Sample':<30} {'Yield':>12}")
print("-" * 44)

# Signal yields
for sample in tmulxy1:
    print(f"{sample:<30} {output[sample]['cutflow'][chan].rows[dphi]['weighted']:>12.2f}")
    print("-" * 44)

# Background yields
dyj = merge_cutflows(output, bkgdyj, chan)
ttj = merge_cutflows(output, bkgttj, chan)
qcd = merge_cutflows(output, bkgqcd, chan)

print(f"{'TTJets':<30} {ttj[dphi]['weighted']:>12.2f}")
print("-" * 44)
print(f"{'DYJ':<30}    {dyj[dphi]['weighted']:>12.2f}")
print("-" * 44)
print(f"{'QCD':<30}    {qcd[dphi]['weighted']:>12.2f}")

## Number of events after all cuts have been applied

In [ ]:
for i in tmulxy1:
    print(i)
    output[i]["cutflow"][ch6].print_table()
    print()

In [ ]:
DYJ = merge_cutflows(output, bkgdyj, ch6)
TTJ = merge_cutflows(output, bkgttj, ch6)
QCD = merge_cutflows(output, bkgqcd, ch6)

backgrounds = {
    "DYJ": bkgdyj,
    "TTJ": bkgttj,
    "QCD": bkgqcd,
}

merged_cf = {}

for name, samples in backgrounds.items():
    merged_cf[name] = merge_cutflows(output, samples, ch6)

for name, cf in merged_cf.items():
    print(f"\n{name}")
    print(f"{'cut name':<20} {'raw':>12} {'weighted':>15}")
    print("-" * 50)

    for cut, vals in cf.items():
        print(
            f"{cut:<20}"
            f"{vals['raw']:>12,.0f}"
            f"{vals['weighted']:>15,.1f}"
        )

In [ ]:
all_bkgs = bkgdyj + bkgttj + bkgqcd
total_bkg = merge_cutflows(output, all_bkgs, ch6)

for cut, vals in total_bkg.items():
    print(
        f"{cut:<20}"
        f"{vals['raw']:>12,.0f}"
        f"{vals['weighted']:>15,.1f}"
    )

In [ ]:
vr = "31"
output = coffea.util.load(f"outputs/bkg_{vr}.coffea")
cf = output["2Mu2E_500GeV_0p25GeV_0p004mm"]["cutflow"][ch6]
cf.print_table()

In [ ]:
# check values for one sample
sample = "2Mu2E_500GeV_0p25GeV_0p004mm"
print(output[sample].keys())
print(output[sample]["cutflow"][ch6].rows["None"])
print(output[sample]["cutflow"][ch6].rows["LJ-LJ dPhi > 2"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ctau = np.array([0.004, 0.04, 0.4, 2.0, 4.0])
def get_acc_eff(output, sample, channel, final_cut="LJ-LJ dPhi > 2"):
    cf = output[sample]["cutflow"][channel].rows

    n_gen = cf["None"]["raw"]
    n_sel = cf[final_cut]["raw"]

    return n_sel / n_gen


acc_eff = []

for s in tmulxy1:
    acc_eff.append(get_acc_eff(output, s, ch6))

acc_eff = np.array(acc_eff)

fig, ax = plt.subplots(figsize=(8,6))

ax.plot(ctau, acc_eff, marker='o', linewidth=2)

ax.set_xscale("log")
ax.set_xlabel("cτ [mm]")
ax.set_ylabel("Acceptance × Efficiency")
ax.set_title("Signal Acceptance × Efficiency vs cτ")

ax.grid(True, which="both", linestyle=":")

plt.tight_layout()
plt.show()  